# Agent Foundations

> Standalone workshop, ~25 min.

Start with a standard LangChain agent (`create_agent()`), then step into Deep Agents.

You'll build:

1. A barebones agent — just a model
2. An `@tool` that calls **OpenAI web search** under the hood
3. An agent with tools
4. A single-turn run with search
5. Deep Agents + **sandbox** — ask it to do math with `execute`
6. Deep Agents + **long-term memory** — remember a fact across threads via `/memories/`
7. Deep Agents + **summarization** — Alex in Boston + commute details until history is compacted
8. Deep Agents + **subagents** — delegate research via `task()` so the main context stays clean

**Needs:** `LANGSMITH_API_KEY` (tracing/sandbox) and `LC_GATEWAY_KEY` (model via gateway). `OPENAI_API_KEY` in `.env` must be the gateway *service* key, not a platform.openai.com key. No Tavily.


## Setup

In [ ]:
import sys
from pathlib import Path
import os

cwd = Path().resolve()
project_root = cwd if (cwd / "utils" / "models.py").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.models import model

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore
from langsmith import uuid7
from langsmith.sandbox import SandboxClient
from deepagents import create_deep_agent
from deepagents.backends import LangSmithSandbox, CompositeBackend, StateBackend, StoreBackend

print("Ready")
print("Model:", getattr(model, "model_name", None) or model)



---

## 1. What is a LangChain agent?

An agent is a **model calling tools in a loop** until the task is done:

1. Model reads the messages
2. Model either answers **or** calls a tool
3. If it called a tool → run the tool → append the result → go back to step 1
4. Stop when the model responds with no more tool calls

`create_agent()` is the standard way to build this in LangChain v1. Under the hood it compiles to a LangGraph — but you don't have to wire the graph yourself.

## 2. Simple agent

Smallest useful agent: **model + system prompt**, no tools. Graph is just `model → end`. Say hello.

In [ ]:
agent = create_agent(
    model=model,
    system_prompt="You are a friendly assistant. Speak like a pirate ",
)

result = agent.invoke({
    "messages": [HumanMessage(content="Write me a poem. ")],
})
print(result["messages"][-1].text)

## 3. OpenAI web search as a tool function

OpenAI's built-in search is normally a **provider dict** (`{"type": "web_search"}`), which runs entirely inside the model call — so LangGraph never adds a `tools` node.

Here we wrap that same OpenAI web search in a normal `@tool`. The agent calls our function; inside it we ask the model with `web_search` bound. Result:

- Still **OpenAI web search** (no Tavily)
- Graph shows the classic **`model` ↔ `tools`** loop

Needs the Responses API (already on in `utils/models.py`).

In [ ]:
@tool(parse_docstring=True)
def openai_web_search(query: str) -> str:
    """Search the web for information using OpenAI web search.

    Args:
        query: Search query to execute.
    """
    # OpenAI runs web_search server-side on this nested call;
    # our @tool wrapper makes it a client-side tool for create_agent.
    search_model = model.bind_tools([{"type": "web_search"}])
    response = search_model.invoke(query)
    return response.text


print("Tool name:", openai_web_search.name)

## 4. Create an agent with tools

Same `create_agent()`, now with tools. Because `openai_web_search` is a real `@tool`, the agent can call it when it needs current facts.

In [ ]:
agent = create_agent(
    model=model,
    tools=[openai_web_search],
    system_prompt=(
        "You are a helpful research assistant. "
        "Use openai_web_search when you need current or specific facts. "
        "Keep answers concise."
    ),
)

## 5. Run a single turn with search

Pass a list of messages. The agent returns updated state — the final answer is the last message.

In [ ]:
result = agent.invoke({
    "messages": [
        HumanMessage(content=" what is the weather in boston")
    ],
})

print(result["messages"][-1].text)

---

## 6. Deep Agents — sandboxing

A default Deep Agent only has an **in-memory filesystem**. A **sandbox backend** also gives an `execute` tool: the agent can run shell/Python **off your laptop**, in an isolated environment.

First demo: ask it to do math. The model should call `execute` (e.g. `python -c '...'`) instead of guessing.

Local calls authenticate with `LC_GATEWAY_KEY` as `OPENAI_API_KEY` against `OPENAI_BASE_URL` (the gateway). For LangSmith-hosted features, add that same *service* key under **LangSmith → Settings → Provider secrets** (do not add a platform OpenAI/Anthropic key), then create a Model configuration pointing at `https://gateway.smith.langchain.com/openai/v1`.

In [ ]:
client = SandboxClient()
ls_sandbox = client.create_sandbox()
backend = LangSmithSandbox(sandbox=ls_sandbox)
print("Sandbox:", ls_sandbox.name)

deep_agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a helpful assistant with a sandbox. "
        "For any math, use the execute tool to run Python. "
        "Do not compute large numbers in your head."
    ),
    backend=backend,
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = deep_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "Use the sandbox to compute 17**24 exactly. "
                    "Print the integer, then tell me the result."
                )
            )
        ]
    },
    config=config,
)
print(result["messages"][-1].text)


---

## 7. Deep Agents — long-term memory

Module 6’s checkpointer is **short-term** (same `thread_id` only). Deep Agents can also keep **long-term** memory: route `/memories/` to a `StoreBackend` so facts survive a **new** `thread_id`.

Ask first (empty store), teach a fact into `/memories/`, then ask again on a fresh thread.

In [ ]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend

store = InMemoryStore()

memory_agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a helpful assistant. Save important facts about the user to "
        "/memories/profile.md. Always check /memories/ before answering questions "
        "about the user."
    ),
    backend=CompositeBackend(
        default=StateBackend(),
        routes={
            "/memories/": StoreBackend(namespace=lambda _rt: ("memories",)),
        },
    ),
    store=store,
    checkpointer=MemorySaver(),
)

# Turn 1 — new thread, empty long-term store
config1 = {"configurable": {"thread_id": str(uuid7())}}
r1 = memory_agent.invoke(
    {"messages": [HumanMessage(content="What do you know about me?")]},
    config=config1,
)
print("Turn 1:", r1["messages"][-1].text)

In [ ]:
# Turn 2 — teach a fact; agent should write it under /memories/
r2 = memory_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "My name is Mike and I live in Boston. "
                    "Save this to /memories/profile.md."
                )
            )
        ]
    },
    config=config1,
)
print("Turn 2:", r2["messages"][-1].text)

In [ ]:
# Turn 3 — brand-new thread_id (no short-term history); store should still have the fact
config2 = {"configurable": {"thread_id": str(uuid7())}}
r3 = memory_agent.invoke(
    {"messages": [HumanMessage(content="What do you know about me? Check /memories/.")]},
    config=config2,
)
print("Turn 3:", r3["messages"][-1].text)

---

## 8. Deep Agents — summarization

Deep Agents includes **conversation summarization** out of the box. When there is nothing left to offload and the **context** approaches ~85% of the model's `max_input_tokens`, older messages are compacted into a summary and the full text is saved under `/conversation_history/`.

Real models often have huge context windows, so that threshold is hard to hit in a short notebook. For this demo we keep the same gateway model but set a **small context window** in its profile — summarization still uses the real model; only the trigger thinks the window is tiny.

We'll tell the agent about Alex in Boston, then pile on commute details until `/conversation_history/` appears. The next cell reads that history file back and asks the agent to recall who he is and how he gets to work.

In [ ]:
small_context_model = model.model_copy(
    update={
        "profile": {
            **(model.profile or {}),
            "max_input_tokens": 4_000,  # demo only — real profile is much larger
        }
    }
)
print(
    "Context window (max_input_tokens):",
    small_context_model.profile.get("max_input_tokens"),
)
print(
    "Summarization trigger (~85%):",
    int(0.85 * small_context_model.profile["max_input_tokens"]),
)

summary_agent = create_deep_agent(
    model=small_context_model,
    system_prompt="You are a helpful assistant. Remember facts the user tells you. Be concise.",
    checkpointer=MemorySaver(),
)

summary_config = {"configurable": {"thread_id": str(uuid7())}}

turns = [
    "I am John and I live in Boston.",
    "I take the Red Line from Davis to Downtown Crossing every weekday.",
    "I leave the apartment around 8:15am and grab coffee at Dunkin on the walk to the T.",
    "The train is usually crowded until Kendall, and I stand by the door with earbuds in.",
    "If the Red Line is delayed I switch to a Bluebike from Cambridge to the office.",
    "I reverse the commute home around 6pm unless I stay late for a deploy.",
]

padding = (
    "Also here are leftover notes from my calendar that you can ignore for now: "
    "standup, one-on-ones, lunch, errands, laundry, groceries, and weekend plans. "
) * 40

summary_result = None
for i, turn in enumerate(turns, start=1):
    content = turn if i == 1 else turn + "\n\n" + padding
    summary_result = summary_agent.invoke(
        {"messages": [HumanMessage(content=content)]},
        config=summary_config,
    )
    history_files = [
        p
        for p in (summary_result.get("files") or {})
        if p.startswith("/conversation_history/")
    ]
    print(f"Turn {i}:", summary_result["messages"][-1].text)
    if history_files:
        print("Summarization wrote:", history_files)
        break
else:
    print("No /conversation_history/ yet — lower max_input_tokens or add more turns.")

In [ ]:
# Read back the summarization artifact + ask the agent to recall
history_files = [
    p
    for p in (summary_result.get("files") or {})
    if p.startswith("/conversation_history/")
]
if not history_files:
    print("No /conversation_history/ file yet — re-run the previous cell.")
else:
    path = history_files[0]
    file_data = summary_result["files"][path]
    content = file_data.get("content", file_data) if isinstance(file_data, dict) else file_data
    if isinstance(content, list):
        content = "\n".join(content)
    print("History file:", path)
    print(str(content)[:1200])
    print("...")

r_final = summary_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Who am I, where do I live, and how do I get to work? Be brief."
            )
        ]
    },
    config=summary_config,
)
print("\nRecall:", r_final["messages"][-1].text)

---

## 9. Deep Agents — subagents

Subagents run in a **separate context**. The main agent delegates with the `task()` tool and only sees the final result — not the subagent's intermediate tool calls. That keeps the main context clean (context quarantine).

We'll give the subagent the same `openai_web_search` tool from earlier, and tell the coordinator to delegate.

In [ ]:
research_subagent = {
    "name": "research-agent",
    "description": "Delegate research tasks. Give one topic at a time.",
    "system_prompt": (
        "You are a research assistant. Use openai_web_search to gather facts. "
        "Return a short summary with source URLs. Limit to 2 search calls."
    ),
    "tools": [openai_web_search],
}

subagent_agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a research coordinator. "
        "Delegate research to research-agent using the task() tool. "
        "Then give the user a brief final answer."
    ),
    subagents=[research_subagent],
    checkpointer=MemorySaver(),
)



In [ ]:
sub_config = {"configurable": {"thread_id": str(uuid7())}}

result = subagent_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "Research what LangChain Deep Agents are, in one short paragraph. "
                    "Use the research-agent via task()."
                )
            )
        ]
    },
    config=sub_config,
)

# Show the tool trail: look for a task() call, then the final answer
for message in result["messages"]:
    name = type(message).__name__
    if name == "AIMessage" and getattr(message, "tool_calls", None):
        for call in message.tool_calls:
            print(f"tool_call: {call.get('name')} -> {call.get('args', {})}")
    elif name == "ToolMessage":
        preview = (message.content if isinstance(message.content, str) else str(message.content))[
            :240
        ].replace("\n", " ")
        print(f"tool_result ({message.name}): {preview}...")

print("\nFinal:", result["messages"][-1].text)

In [ ]:
# Always stop the sandbox when you're done with this section
if "client" in globals() and "ls_sandbox" in globals():
    client.delete_sandbox(ls_sandbox.name)
    print("Deleted", ls_sandbox.name)
else:
    print("No sandbox to delete — run the create cell first.")